# Phase 3, experiment 1: degree features

Goal: test whether adding graph degree features improves on the baseline.

**The one change in this experiment:** three new features are added to the 165 raw features.

- `in_degree`: number of edges pointing into the transaction (how many earlier transactions it spends from)
- `out_degree`: number of edges leaving the transaction (how many later transactions spend its outputs)
- `total_degree`: `in_degree + out_degree`

**Decision: count all edges.** About 25% of the edges touch a transaction that is in
neither train nor test. These edges are still counted, because they are real payment flows and the
competition description says degree reflects real transaction behaviour.

Everything else is identical to `02_baseline.ipynb`: same split (train on steps 1 to 21, validate on
22 to 35), same model (LightGBM defaults, seed 42), no class weighting. The baseline is retrained here
so the two models can be compared side by side.

Pipeline order: load → features → split → train → validate → predict → save CSV.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("train:", train.shape, " test:", test.shape, " edges:", edges.shape)

train: (141772, 168)  test: (15329, 168)  edges: (234355, 2)


## 2. Degree features

Each row of the edge list is `(txId1, txId2)`: an output of `txId1` is spent by `txId2`.

- A transaction's **out-degree** is the number of times it appears as `txId1`.
- A transaction's **in-degree** is the number of times it appears as `txId2`.

`value_counts()` counts how many times each ID appears in a column. The degrees do not use labels,
so there is no leakage, and they are computed the same way for train and test.

In [ ]:
# Count over ALL edges, including edges whose other endpoint is not in train or test
out_deg = edges["txId1"].value_counts()   # index = txId, value = number of outgoing edges
in_deg = edges["txId2"].value_counts()    # index = txId, value = number of incoming edges

def add_degree_features(df):
    
    df = df.copy()
    df["in_degree"] = df["txId"].map(in_deg).fillna(0).astype(int)
    df["out_degree"] = df["txId"].map(out_deg).fillna(0).astype(int)
    df["total_degree"] = df["in_degree"] + df["out_degree"]
    return df

train = add_degree_features(train)
test = add_degree_features(test)

degree_cols = ["in_degree", "out_degree", "total_degree"]
train[degree_cols].describe().round(2)

,in_degree,out_degree,total_degree
count,141772.00,141772.00,141772.00
mean,1.15,1.15,2.30
std,3.57,1.96,4.05
min,0.00,0.00,1.00
25%,0.00,1.00,1.00
50%,1.00,1.00,2.00
75%,1.00,1.00,2.00
max,284.00,472.00,473.00


In [3]:
# Two feature sets: the baseline (raw features only) and the new one (raw + degree)
raw_cols = [c for c in train.columns if c.startswith("feat_")]
new_cols = raw_cols + degree_cols
print("Baseline features:", len(raw_cols), " New features:", len(new_cols))

# Keep only labeled rows for training and validation (1 = illicit, 0 = licit)
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

Baseline features: 165  New features: 168


## 3. Time-based split (same as the baseline)

In [ ]:
TRAIN_LAST_STEP = 21   # train on steps 1-21, validate on steps 22-35

tr = labeled[labeled["time_step"] <= TRAIN_LAST_STEP]
va = labeled[labeled["time_step"] > TRAIN_LAST_STEP]
y_tr, y_va = tr["label"], va["label"]

print("Train rows:", len(tr), " Validation rows:", len(va))

Train rows: 19530  Validation rows: 11705


## 4. Train both models

The same model settings are used twice. The only difference is the list of feature columns.

In [5]:
base_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
base_model.fit(tr[raw_cols], y_tr)

new_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
new_model.fit(tr[new_cols], y_tr)

,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


## 5. Validate and compare

In [6]:
base_pred = base_model.predict_proba(va[raw_cols])[:, 1]   # column 1 = probability of illicit
new_pred = new_model.predict_proba(va[new_cols])[:, 1]

base_auc = roc_auc_score(y_va, base_pred)
new_auc = roc_auc_score(y_va, new_pred)

print("Baseline validation AUC:     ", round(base_auc, 4))
print("With degree, validation AUC: ", round(new_auc, 4))
print("Change:                      ", round(new_auc - base_auc, 4))

Baseline validation AUC:      0.9148
With degree, validation AUC:  0.8772
Change:                       -0.0376


In [7]:
# AUC per validation time step for both models, to see where the change helps or hurts.
# Steps 24, 29, 31 and 32 were the weak ones for the baseline.
va_result = va[["time_step", "label"]].copy()
va_result["base_pred"] = base_pred
va_result["new_pred"] = new_pred

rows = []
for step, group in va_result.groupby("time_step"):
    rows.append({
        "time_step": step,
        "n_illicit": group["label"].sum(),
        "base_auc": roc_auc_score(group["label"], group["base_pred"]),
        "new_auc": roc_auc_score(group["label"], group["new_pred"]),
    })
per_step = pd.DataFrame(rows).set_index("time_step")
per_step["change"] = per_step["new_auc"] - per_step["base_auc"]
per_step.round(4)

,n_illicit,base_auc,new_auc,change
time_step,,,,
22,158,0.9991,0.9993,0.0002
23,53,0.9591,0.9532,-0.0058
24,137,0.7866,0.7426,-0.0441
25,118,0.9997,0.9998,0.0000
26,96,0.9312,0.9202,-0.0110
27,24,0.9995,0.9995,0.0000
28,85,1.0000,1.0000,0.0000
29,329,0.7562,0.6943,-0.0619
30,83,0.9886,0.9897,0.0011


In [ ]:
# Rank = position among all 168 features
# importance = number of splits that use the feature
importance = pd.Series(new_model.feature_importances_, index=new_cols).sort_values(ascending=False)
rank = pd.Series(range(1, len(importance) + 1), index=importance.index)
pd.DataFrame({"importance": importance[degree_cols], "rank": rank[degree_cols]})

,importance,rank
in_degree,12,73
out_degree,3,128
total_degree,10,84


## 6. Retrain on all labeled rows and predict the test set

Only submit this file if the validation AUC above improved on the baseline.

In [9]:
final_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
final_model.fit(labeled[new_cols], labeled["label"])

test_pred = final_model.predict_proba(test[new_cols])[:, 1]
print("Test predictions: min", round(test_pred.min(), 4), " mean", round(test_pred.mean(), 4),
      " max", round(test_pred.max(), 4))

Test predictions: min 0.0  mean 0.0445  max 0.9997


## 7. Save the submission

In [ ]:
submission = pd.DataFrame({"index": test["index"], "target": test_pred})

assert len(submission) == 15329, "wrong number of rows"
assert list(submission.columns) == ["index", "target"], "wrong columns"
assert submission["target"].notna().all(), "NaN in predictions"
assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

SUB_PATH = "../submissions/sub02_degree.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved", SUB_PATH)

Saved ../submissions/sub02_degree.csv


## Results

- **Validation AUC, baseline vs with degree features:** 0.9148 for the baseline and 0.8772 with the
  three degree features, a drop of 0.0376. Adding degree features made validation worse.
- **Per-step changes (did the weak steps 24, 29, 31, 32 improve?):** No, they got worse, and almost all
  of the drop comes from them: step 24 fell by 0.044, step 29 by 0.062, step 31 by 0.070 and step 32 by
  0.075. Eight of the other ten steps changed by less than 0.005 (step 23 fell by 0.006 and step 26 by 0.011).
- **Did the model use the degree features?** Barely. `in_degree` was used for 12 splits (rank 73 of 168),
  `total_degree` for 10 (rank 84) and `out_degree` for 3 (rank 128). For comparison, the most-used
  feature in the baseline was used for 298 splits.
- **Observation:** three features that the model hardly uses still moved the weak-step scores a lot.
  This suggests the scores on those steps are unstable (sensitive to small changes in the model), but
  that has not been tested yet.
- **Decision: keep the degree features?** _to decide_
- **Public leaderboard AUC for `sub02_degree.csv`:** 0.94770, against 0.94660 for the baseline
  (`sub01_baseline.csv`), an increase of 0.0011.
- **Validation and leaderboard disagree:** validation fell by 0.0376 but the public leaderboard rose by
  0.0011. The leaderboard change is small enough to be noise (it is measured on 30% of the test set), but
  the large validation drop clearly did not carry over to the test period. So the current validation
  split, whose score is dominated by the unstable weak steps, is not yet a reliable guide for choosing
  between models.
